# 👁️ Arogya Sahayak — Deep Learning Scanner 04: Diabetic Retinopathy Detection
### Microvascular Lesion Staging, Retinal Grad-CAM & TensorFlow.js Edge Export

**Project:** Arogya Sahayak — Offline-First Multilingual Rural Telemedicine Assistant  
**Institution:** ABIT Cuttack, Affiliated to BPUT Odisha  
**Student Developers:** Rahul Prasad Das (2301206094), Purnachandra Swain (2301206093), Rohit Kumar Thakur (2301206103), Ritik Kumar Swain (2301206102)  
**Project Guide:** Prof. Rajesh Kumar Sahoo, Assistant Professor, Dept. of CSE  
**Constraint:** 100% Free Tiers (Google Colab T4 GPU, Kaggle Datasets, Zero Cost)  

---

### 🎯 Clinical Objectives:
1. **Microvascular Retinopathy Staging:** Classify macula-centered retinal fundus photographs across 4 clinical severity grades:
   - **`0: No Retinopathy (Normal)`** — Intact retinal vasculature, sharp optic disc margin, clear macula without microaneurysms or exudates.
   - **`1: Mild Non-Proliferative (NPDR)`** — Isolated retinal microaneurysms, minimal dot hemorrhages.
   - **`2: Moderate NPDR`** — Multiple blot hemorrhages, hard lipid exudates, cotton-wool ischemic spots across vascular arcades.
   - **`3: Severe / Proliferative (PDR)`** — Extensive intraretinal microvascular abnormalities (IRMA), venous beading, neovascularization prone to vitreous hemorrhage.
2. **MobileNetV2 Deep Transfer Learning:** Train on Gaussian-filtered 224x224 retinal fundus images with progressive unfreezing.
3. **Clinical Evaluation:** Quantify diagnostic Sensitivity (Recall), Specificity, Multiclass AUC-ROC curves, and Confusion Matrices.
4. **Retinal Grad-CAM Explainability:** Extract authentic partial derivatives from the final convolutional layer (`Conv_1`) to verify that the network focuses on microaneurysms and hemorrhagic lesions rather than optic disc glare or artifacts.
5. **Edge Deployment:** Export to sub-12 MB TensorFlow.js graph weights for zero-latency in-browser mobile triage.

---
## 1. Environment Setup & Hardware Acceleration
Verify GPU availability (NVIDIA Tesla T4 on free Colab) and install necessary packages.

In [ ]:
# Verify GPU accelerator
!nvidia-smi

# Install dependencies (quiet mode)
!pip install -q kagglehub tensorflowjs scikit-learn seaborn matplotlib pandas pillow

import os
import glob
import json
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Devices: {tf.config.list_physical_devices('GPU')}")

---
## 2. Dataset Acquisition (APTOS 2019 Gaussian-Filtered 224x224)
We use the benchmark Diabetic Retinopathy dataset (`sovitrath/diabetic-retinopathy-224x224-gaussian-filtered`), featuring 3,662 pre-processed, high-contrast retinal fundus scans across all stages of microvascular disease.

In [ ]:
from google.colab import files
import kagglehub

dataset_raw_path = "./retinopathy_dataset"
os.makedirs(dataset_raw_path, exist_ok=True)

# Check if kaggle.json is present, or upload it
if not os.path.exists('kaggle.json'):
    print("🔑 Upload your kaggle.json file (if you have one) or cancel to try direct download:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('kaggle.json'):
            shutil.move(fn, 'kaggle.json')

if os.path.exists('kaggle.json'):
    with open('kaggle.json') as f:
        creds = json.load(f)
    os.environ['KAGGLE_USERNAME'] = creds.get('username', '')
    os.environ['KAGGLE_KEY'] = creds.get('key', '')
    print(f"✅ Kaggle Token Authenticated as: {creds.get('username', 'user')}")
    print("📥 Downloading Diabetic Retinopathy dataset via Kaggle CLI...")
    !kaggle datasets download -d sovitrath/diabetic-retinopathy-224x224-gaussian-filtered --unzip -p {dataset_raw_path}
else:
    print("📥 Downloading Diabetic Retinopathy dataset via KaggleHub...")
    downloaded_dir = kagglehub.dataset_download("sovitrath/diabetic-retinopathy-224x224-gaussian-filtered")
    dataset_raw_path = downloaded_dir

print(f"\n📁 Dataset root: {dataset_raw_path}")
for root, dirs, fnames in os.walk(dataset_raw_path):
    depth = root[len(dataset_raw_path):].count(os.sep)
    if depth < 3:
        print(f"{'  ' * depth}📁 {os.path.basename(root)}/ ({len(fnames)} files)")

---
## 3. Data Ingestion & Clinical Label Standardization
We standardize the APTOS 5-tier classification into the 4 diagnostic severity stages defined in Arogya Sahayak's `scanConfig.ts`:
- `0 - No_DR` → `0: No Retinopathy (Normal)`
- `1 - Mild` → `1: Mild Non-Proliferative (NPDR)`
- `2 - Moderate` → `2: Moderate NPDR`
- `3 - Severe` & `4 - Proliferate_DR` → `3: Severe / Proliferative (PDR)`

In [ ]:
CLASS_MAP = {
    '0 - no_dr': {'idx': 0, 'name': 'No Retinopathy (Normal)'},
    'no_dr': {'idx': 0, 'name': 'No Retinopathy (Normal)'},
    'nodr': {'idx': 0, 'name': 'No Retinopathy (Normal)'},
    '1 - mild': {'idx': 1, 'name': 'Mild Non-Proliferative (NPDR)'},
    'mild': {'idx': 1, 'name': 'Mild Non-Proliferative (NPDR)'},
    '2 - moderate': {'idx': 2, 'name': 'Moderate NPDR'},
    'moderate': {'idx': 2, 'name': 'Moderate NPDR'},
    '3 - severe': {'idx': 3, 'name': 'Severe / Proliferative (PDR)'},
    'severe': {'idx': 3, 'name': 'Severe / Proliferative (PDR)'},
    '4 - proliferate_dr': {'idx': 3, 'name': 'Severe / Proliferative (PDR)'},
    'proliferate_dr': {'idx': 3, 'name': 'Severe / Proliferative (PDR)'},
    'proliferatedr': {'idx': 3, 'name': 'Severe / Proliferative (PDR)'},
}
CLASS_NAMES = [
    'No Retinopathy (Normal)',
    'Mild Non-Proliferative (NPDR)',
    'Moderate NPDR',
    'Severe / Proliferative (PDR)'
]

# Search for all image files
extensions = ('*.png', '*.jpg', '*.jpeg')
all_image_paths = []
for ext in extensions:
    all_image_paths.extend(glob.glob(os.path.join(dataset_raw_path, "**", ext), recursive=True))

records = []
for p in all_image_paths:
    parent = os.path.basename(os.path.dirname(p)).lower().replace('_', ' ').strip()
    matched = None
    for k, v in CLASS_MAP.items():
        if k in parent or parent in k or k.replace(' ', '') in parent.replace(' ', ''):
            matched = v
            break
    if matched:
        records.append({
            'filepath': p,
            'label_idx': matched['idx'],
            'label_name': matched['name']
        })

df_all = pd.DataFrame(records)
print(f"Total Retinal Fundus Scans Ingested: {len(df_all):,}")
print("\nClass Distribution across Dataset:")
print(df_all['label_name'].value_counts())

---
## 4. Stratified Dataset Partitioning (70% Train, 15% Val, 15% Test)
Preserves proportional clinical severity ratios across training, validation, and untouched evaluation sets.

In [ ]:
# Stratified 70/15/15 split
df_train, df_temp = train_test_split(
    df_all,
    test_size=0.30,
    random_state=SEED,
    stratify=df_all['label_idx'].values
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=df_temp['label_idx'].values
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print(f"Training Cohort:   {len(df_train):,} scans ({len(df_train)/len(df_all):.1%})")
print(f"Validation Cohort: {len(df_val):,} scans ({len(df_val)/len(df_all):.1%})")
print(f"Test Cohort:       {len(df_test):,} scans ({len(df_test)/len(df_all):.1%})")

---
## 5. Visualizing Retinal Fundus Across Diabetic Retinopathy Stages
Examine normal vascular geometry vs progressive microaneurysms, hemorrhages, and neovascularization.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for c_idx, c_name in enumerate(CLASS_NAMES):
    sample = df_train[df_train['label_idx'] == c_idx].iloc[0]
    img = Image.open(sample['filepath']).convert('RGB')
    axes[c_idx].imshow(img)
    axes[c_idx].set_title(f"Stage {c_idx}:\n{c_name}", fontsize=10, fontweight='bold')
    axes[c_idx].axis('off')
plt.suptitle('Representative Fundus Scans Across Diabetic Retinopathy Severity Stages', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Input Pipeline & Fundus Data Augmentation
Standardize to $224 \times 224 \times 3$ with MobileNetV2 $[-1, 1]$ normalization.  
Augmentation leverages horizontal and vertical flips, subtle rotations, and brightness/contrast adjustments tailored to fundus cameras.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4

retinopathy_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.08, fill_mode='nearest'),
    layers.RandomZoom((-0.05, 0.05), fill_mode='nearest'),
    layers.RandomBrightness(0.06),
    layers.RandomContrast(0.06),
], name="retinopathy_fundus_augmentation")

def load_and_preprocess_retinopathy(path, label):
    img_raw = tf.io.read_file(path)
    img = tf.image.decode_png(img_raw, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)  # MobileNetV2 [-1, 1] scaling
    one_hot = tf.one_hot(label, depth=NUM_CLASSES)
    return img, one_hot

def create_retinopathy_dataset(df, is_training=False):
    paths = df['filepath'].values
    labels = df['label_idx'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess_retinopathy, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        ds = ds.shuffle(buffer_size=1024, seed=SEED)
        ds = ds.map(lambda x, y: (retinopathy_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_retinopathy_dataset(df_train, is_training=True)
val_ds   = create_retinopathy_dataset(df_val, is_training=False)
test_ds  = create_retinopathy_dataset(df_test, is_training=False)

# Class weight calculation to counterbalance minority classes
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df_train['label_idx']),
    y=df_train['label_idx'].values
)
class_weights_dict = {i: float(w) for i, w in enumerate(weights)}
print("Computed Balanced Class Weights:")
for k, v in class_weights_dict.items():
    print(f"  Class {k} ({CLASS_NAMES[k]}): weight = {v:.3f}")

---
## 7. Model Architecture (MobileNetV2 Retinopathy Classifier)
We use MobileNetV2 with Global Average Pooling, Batch Normalization, Dropout (0.35), and a 4-class Softmax classification head.

In [ ]:
def build_retinopathy_model():
    inputs = keras.Input(shape=(224, 224, 3), name="input_retinal_fundus")
    
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_tensor=inputs
    )
    base_model.trainable = False  # Freeze for Phase 1
    
    x = base_model.output
    x = layers.GlobalAveragePooling2D(name="retinopathy_global_pool")(x)
    x = layers.BatchNormalization(name="retinopathy_bn")(x)
    x = layers.Dense(128, activation='relu', name="retinopathy_dense_128")(x)
    x = layers.Dropout(0.35, name="retinopathy_dropout")(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name="retinopathy_probabilities")(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs, name="Arogya_Retinopathy_MobileNetV2")
    return model, base_model

model, base_model = build_retinopathy_model()
model.summary()

---
## 8. Phase 1 Training: Feature Adaptation (Frozen Backbone)
Train custom dense head with Adam ($10^{-3}$) and categorical crossentropy loss.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

callbacks_phase1 = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6, verbose=1)
]

print("Starting Phase 1 Training (Frozen Backbone)...")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weights_dict,
    callbacks=callbacks_phase1,
    verbose=1
)

---
## 9. Phase 2 Training: Fine-Tuning Upper Convolutional Layers
Unfreeze the top 35 layers of MobileNetV2 (including bottleneck block 16 and final conv `Conv_1`) and fine-tune with a low learning rate ($10^{-5}$) to adapt filters to retinal microaneurysms and hard exudates.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-35]:
    layer.trainable = False

trainable_params = sum(np.prod(p.shape) for p in model.trainable_weights)
print(f"Trainable parameters for fine-tuning: {trainable_params:,}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

callbacks_phase2 = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1)
]

print("Starting Phase 2 Fine-Tuning...")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    class_weight=class_weights_dict,
    callbacks=callbacks_phase2,
    verbose=1
)

In [ ]:
acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy', color='#10b981', lw=2)
plt.plot(val_acc, label='Validation Accuracy', color='#3b82f6', lw=2)
plt.axvline(x=len(history_phase1.history['accuracy'])-1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('Retinopathy Classifier: Accuracy Trajectory', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss', color='#ef4444', lw=2)
plt.plot(val_loss, label='Validation Loss', color='#f59e0b', lw=2)
plt.axvline(x=len(history_phase1.history['loss'])-1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('Retinopathy Classifier: Loss Trajectory', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Categorical Crossentropy')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Clinical Evaluation on Held-Out Test Cohort
Assess diagnostic sensitivity, specificity, precision, and F1-score across all 4 progressive retinopathy stages.

In [ ]:
print("Evaluating on untouched Test Set...")
y_true = []
y_pred_probs = []

for batch_imgs, batch_labels in test_ds:
    probs = model.predict(batch_imgs, verbose=0)
    y_true.extend(np.argmax(batch_labels.numpy(), axis=1))
    y_pred_probs.extend(probs)

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification report
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, output_dict=True)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix: Held-Out Diabetic Retinopathy Test Scans', fontweight='bold')
plt.xlabel('Predicted Clinical Stage')
plt.ylabel('True Severity Stage')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

---
## 11. Retinal Grad-CAM Saliency Localization
Verify that the model activates on microvascular hemorrhages, hard exudates, and vascular arcades via the final convolutional layer (`Conv_1`).

In [ ]:
def make_retinopathy_gradcam_heatmap(img_array, model, last_conv_layer_name="Conv_1", pred_index=None):
    grad_model = keras.models.Model(
        inputs=[model.inputs],
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

# Visualize Grad-CAM on 4 test samples
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for idx in range(4):
    sample_path = df_test.iloc[idx]['filepath']
    true_label = df_test.iloc[idx]['label_name']
    
    img_pil = Image.open(sample_path).convert('RGB').resize(IMG_SIZE)
    img_arr = np.array(img_pil)
    img_tensor = preprocess_input(np.expand_dims(img_arr.astype(np.float32), axis=0))
    
    pred = model.predict(img_tensor, verbose=0)[0]
    pred_idx = np.argmax(pred)
    heatmap = make_retinopathy_gradcam_heatmap(img_tensor, model)
    
    axes[0, idx].imshow(img_pil)
    axes[0, idx].set_title(f"True: {true_label}\nPred: {CLASS_NAMES[pred_idx]} ({pred[pred_idx]:.1%})", fontsize=10)
    axes[0, idx].axis('off')
    
    heatmap_resized = np.uint8(255 * heatmap)
    axes[1, idx].imshow(img_pil)
    axes[1, idx].imshow(heatmap_resized, cmap='jet', alpha=0.45)
    axes[1, idx].set_title(f"Conv_1 Grad-CAM Localization", fontsize=10, fontweight='bold', color='navy')
    axes[1, idx].axis('off')

plt.suptitle('Clinical Grad-CAM Saliency Verification on Retinal Microvascular Lesions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 12. TensorFlow.js Web Export (Direct In-Memory Exporter)
Export Keras model directly into TensorFlow.js format (`model.json` + `group1-shard1of1.bin`, ~9.5 MB) using pure Python memory buffers.

In [ ]:
EXPORT_DIR = "./arogya_retinopathy_tfjs/web_model"
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Extract Keras topology
try:
    topology = json.loads(model.to_json())
except Exception as e:
    print(f"Topology warning: {e}")
    topology = {}

# 2. Extract weights into binary shard
manifest_weights = []
raw_bytes = bytearray()
for w in model.weights:
    w_name = w.path if hasattr(w, 'path') else w.name
    if isinstance(w_name, str) and w_name.endswith(':0'):
        w_name = w_name[:-2]
    w_arr = w.numpy().astype(np.float32)
    manifest_weights.append({
        "name": str(w_name),
        "shape": list(w_arr.shape),
        "dtype": "float32"
    })
    raw_bytes.extend(w_arr.tobytes())

shard_filename = "group1-shard1of1.bin"
shard_path = os.path.join(EXPORT_DIR, shard_filename)
with open(shard_path, "wb") as f:
    f.write(raw_bytes)
print(f"  ✅ Saved weight shard: {shard_filename} ({len(raw_bytes) / (1024*1024):.2f} MB)")

# 3. Write model.json
tfjs_manifest = {
    "format": "layers-model",
    "generatedBy": f"keras v{tf.__version__}",
    "convertedBy": "Arogya Sahayak Direct Exporter",
    "modelTopology": topology,
    "weightsManifest": [{
        "paths": [shard_filename],
        "weights": manifest_weights
    }]
}
model_json_path = os.path.join(EXPORT_DIR, "model.json")
with open(model_json_path, 'w', encoding='utf-8') as f:
    json.dump(tfjs_manifest, f, indent=2)
print("  ✅ Saved model.json topology.")

---
## 13. Model Card & Clinical Audit Metadata
Generate Mitchell et al. `model_card.json` and `metrics.json` recording sensitivity, specificity, and ethical safety boundaries.

In [ ]:
import datetime

class_metrics = []
for c_idx, c_name in enumerate(CLASS_NAMES):
    sens = float(cm[c_idx, c_idx] / cm[c_idx].sum())
    class_metrics.append({
        "index": c_idx,
        "label": c_name,
        "sensitivity": sens,
        "precision": float(report[c_name]['precision']),
        "f1_score": float(report[c_name]['f1-score'])
    })

overall_accuracy = float(np.sum(np.diag(cm)) / np.sum(cm))

model_card = {
    "schema_version": "1.0.0",
    "model_name": "Arogya Sahayak Diabetic Retinopathy MobileNetV2 Classifier",
    "scanner_id": "retinopathy",
    "modality": "Retinal Fundus Photography (Macula-Centered)",
    "input_dimensions": [224, 224, 3],
    "preprocessing": "MobileNetV2 scale [-1, 1]",
    "classes": class_metrics,
    "metrics": {
        "overall_test_accuracy": float(overall_accuracy),
        "macro_avg_f1": float(report['macro avg']['f1-score']),
        "weighted_avg_f1": float(report['weighted avg']['f1-score'])
    },
    "calibrated_threshold": 0.52,
    "last_conv_layer": "Conv_1",
    "training_metadata": {
        "date": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "institution": "ABIT Cuttack / BPUT Odisha",
        "developers": ["Rahul Prasad Das", "Purnachandra Swain", "Rohit Kumar Thakur", "Ritik Kumar Swain"],
        "guide": "Prof. Rajesh Kumar Sahoo",
        "dataset_source": "APTOS 2019 Blindness Detection / Gaussian-Filtered Fundus",
        "training_samples": len(df_train),
        "validation_samples": len(df_val),
        "test_samples": len(df_test),
        "hardware": "Google Colab Tesla T4 GPU (Free Tier)"
    },
    "clinical_safety_notice": "SCREENING AID ONLY. Not intended as a substitute for dilated indirect ophthalmoscopy or optical coherence tomography (OCT) performed by a certified vitreoretinal specialist. Any detected NPDR or PDR requires urgent clinical dilated retinal exam."
}

model_card_file = os.path.join(EXPORT_DIR, "model_card.json")
with open(model_card_file, 'w', encoding='utf-8') as f:
    json.dump(model_card, f, indent=2)

metrics_file = os.path.join(EXPORT_DIR, "metrics.json")
with open(metrics_file, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

print("Model Card & Audit Metrics JSON generated successfully.")
print(json.dumps(model_card, indent=2))

---
## 14. Package & Download Model for Arogya Sahayak Integration
Packages the exported web model into `arogyasahayak_retinopathy_model.zip` and triggers an automatic browser download.

In [ ]:
import zipfile
from google.colab import files

zip_filename = "arogyasahayak_retinopathy_model.zip"
print(f"Creating deployment archive: {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, filenames in os.walk(EXPORT_DIR):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, EXPORT_DIR)
            zipf.write(fpath, arcname)
            print(f"  Added: {arcname}")

zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"\nPackage ready! Size: {zip_size_mb:.2f} MB")
print("Initiating browser download to your local machine...")
files.download(zip_filename)

print("""
==========================================================================
✅ DEPLOYMENT INSTRUCTIONS:
1. Extract 'arogyasahayak_retinopathy_model.zip'.
2. Copy the extracted files into your Arogya Sahayak repository at:
   public/models/retinopathy/
   (Files needed: model.json, group1-shard1of1.bin, model_card.json, metrics.json)
3. Done! Arogya Sahayak will now run real deep-learning Diabetic Retinopathy grading!
==========================================================================
""")